# Test 3 Different Models

In [6]:
import ollama
import chromadb
import time
from datetime import datetime

In [17]:
# Models to test
MODELS_TO_TEST = [
    'qwen2.5:14b',     # Qwen 2.5 14B - best quality for 16GB RAM
    'llama3.1:8b',      # Meta's Llama 3.1 8B - efficient/fast, provider for instruction following, midsize  
    'mistral:7b'        # Mistral 7B - strong instruction following
]

In [3]:
TEST_PROMPTS = [
    {
        'type': 'tweet',
        'prompt': "Write a tweet about clinical trial diversity and inclusion, in the professional yet accessible voice of AbbVie. Include relevant hashtags.",
        'context': 'AbbVie tweets are concise, professional, use hashtags, and often include a call to action or link to more information.'
    },

    {
        'type': 'tweet',
        'prompt': "Write a motivational tweet for patients living with chronic illness, in the professional yet accessible voice of AbbVie. Include relevant hashtags.",
        'context': 'AbbVie tweets are concise, professional, use hashtags, and often include a call to action or link to more information.'
    },

    {
        'type': 'press_release',
        'prompt': "Write the opening paragraph of a press release announcing positive clinical trial results for a new treatment. Use AbbVie's professional, authoritative tone.",
        'context': 'AbbVie press releases are formal, detailed, evidence-based, and emphasize patient impact and scientific rigor.'
    },

    {
        'type': 'press_release',
        'prompt': "Write the opening paragraph of a press release announcing a new partnership between AbbVie and a leading research institution. Use AbbVie's professional, authoritative tone.",
        'context': 'AbbVie press releases are formal, detailed, evidence-based, and emphasize patient impact and scientific rigor.'
    }
]

In [4]:
def test_model(client, model_name, test_prompt):
    """Test a single model with a prompt"""
    try:
        start_time = time.time()
        response = client.generate(
            model=model_name, 
            prompt=f"{test_prompt['context']}\n\n{test_prompt['prompt']}",
            options={
                'temperature': 0.7,  # Adjust for creativity
                'top_p': 0.9,        # Adjust for diversity
                'max_tokens': 300 if test_prompt['type'] == 'tweet' else 500
            }
        )
        
        elapsed_time = time.time() - start_time
        generated_text = response['response']

        return {
            'success': True,
            'text': generated_text,
            'time' : elapsed_time,
            'tokens': response.get('eval_count', 0)
        }
        
    except Exception as e:
        return {
            'success': False,
            'error': str(e)
        }


In [5]:
def evaluate_output(generated_text, test_type):
    """Basic evaluation of the generated text based on type"""

    evaluation = {
        'appropriate_length': False,
        'has_hashtags': False,
        'professional_tone': False
    }

    text = generated_text.lower()

    if test_type == 'tweet':
        evaluation['appropriate_length'] = 50 < len(generated_text) < 280
    else:
        evaluation['appropriate_length'] = len(generated_text) > 200

    if test_type == 'tweet':
        evaluation['has_hashtags'] = '#' in generated_text

    professional_words = ['patient', 'clinical', 'treatment', 'research', 'partnership', 'results', 'trial']
    evaluation['professional_tone'] = any(word in text for word in professional_words)

    score = sum(evaluation.values()) / len(evaluation) * 100

    return evaluation, score

In [15]:
def load_sample_abbvie_content():
    """Load data from Chroma database for reference"""
    try:
        client = chromadb.PersistentClient(path='./chroma_db')
        collection = client.get_collection('abbvie_social_media')

        tweets = collection.get(
            where={"content_type": "tweet"},
            limit=1244
        )

        if tweets['metadata']:
            tweet_data = list(zip(tweets['documents'], tweets['metadatas']))
            sorted_tweets = sorted(tweet_data, 
                                   key=lambda x: x[1].get('total_engagement', 0), 
                                   reverse=True)
            top_tweets = [doc for doc, meta in sorted_tweets[:10]]
        else:
            top_tweets = []

        press_release = collection.get(
            where={"content_type": "press_release"}
        )
        
        sample_pr = [doc[:10] for doc in press_release['documents']] if press_release['documents'] else []

        return top_tweets, sample_pr
    except Exception as e:
        print(f"Error loading sample content: {e}")
        return [], []

## Testing

In [9]:
client = ollama.Client()
model_info = client.list()

In [10]:
model_info

ListResponse(models=[Model(model='gpt-oss:20b', modified_at=datetime.datetime(2026, 1, 15, 13, 11, 25, 986877, tzinfo=TzInfo(-18000)), digest='17052f91a42e97930aa6e28a6c6c06a983e6a58dbb00434885a0cf5313e376f7', size=13793441244, details=ModelDetails(parent_model='', format='gguf', family='gptoss', families=['gptoss'], parameter_size='20.9B', quantization_level='MXFP4')), Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 1, 15, 12, 55, 41, 47249, tzinfo=TzInfo(-18000)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M')), Model(model='deepseek-r1:1.5b', modified_at=datetime.datetime(2025, 11, 9, 20, 34, 40, 43780, tzinfo=TzInfo(-18000)), digest='e0979632db5a88d1a53884cb2a941772d10ff5d055aabaa6801c4e36f3a6c2d7', size=1117322768, details=ModelDetails(parent_model='', format='gguf', family='qwen

In [22]:
top_tweets, sample_press_releases = load_sample_abbvie_content()
if top_tweets:
    print(f"Loaded {len(top_tweets)} sample tweets from Chroma database.")
    print(f"Loaded {len(sample_press_releases)} sample press releases from Chroma database.")

results = []

for model_name in MODELS_TO_TEST:
    print(f"\nTesting model: {model_name}")

    model_results = {
            'model': model_name,
            'tests': [],
            'average_time': 0,
            'average_score': 0
        }
    
    for i, test_prompt in enumerate(TEST_PROMPTS):
        print(f"  Test {i+1}/{len(TEST_PROMPTS)}: {test_prompt['type'][:50]} generation")
        print(f"Prompt: {test_prompt['prompt'][:100]}...")

        result = test_model(client, model_name, test_prompt)

        if result['success']:
            print(f" Generated in {result['time']:.2f}s ({result['tokens']} tokens)")

            output_preview = result['text'][:750] + ('...' if len(result['text']) > 750 else '')
            for line in output_preview.split('\n'):
                print(f"   {line}")
            
            evaluation, score = evaluate_output(result['text'], test_prompt['type'])
            print(f" Evaluation: {evaluation}, Score: {score:.2f}%")

            model_results['tests'].append({
                'prompt': test_prompt['prompt'],
                'type': test_prompt['type'],
                'generated_text': result['text'],
                'time': result['time'],
                'tokens': result['tokens'],
                'evaluation': evaluation,
                'score': score
            })
        else:
            print(f" Failed: {result['error']}")
    
    if model_results['tests']:
        model_results['average_time'] = sum(test['time'] for test in model_results['tests']) / len(model_results['tests'])
        model_results['average_score'] = sum(test['score'] for test in model_results['tests']) / len(model_results['tests'])
    
    results.append(model_results)

print("Summary - Model Comparison")

if results:
    print(f"{'Model':<20} {'Avg Time (s)':<15} {'Avg Score (%)':<15} {'Recommendation'}")

    results_sorted = sorted(results, key=lambda x: x['average_score'], reverse=True)

    for i, r in enumerate(results_sorted):
        recommendation = "Best" if i == 0 else "Good" if r['average_score'] > 70 else "Okay"
        print(f"{r['model']:<20}, {r['average_time']:<15.2f}, {r['average_score']:<15.2f}, {recommendation}")

    print("Recommendation")
    best_model = results_sorted[0]
    print(f"""Based on the evaluation, I recommend using {best_model['model']} for AbbVie's social media content generation. 
          It achieved the highest average score of {best_model['average_score']:.2f}% while maintaining a reasonable generation time of {best_model['average_time']:.2f} seconds per prompt. 
          This model demonstrated strong performance in generating professional, engaging, and appropriately formatted content that aligns well with AbbVie's brand voice and communication goals.""")

Error loading sample content: 'metadata'

Testing model: qwen2.5:14b
  Test 1/4: tweet generation
Prompt: Write a tweet about clinical trial diversity and inclusion, in the professional yet accessible voice...
 Generated in 22.82s (64 tokens)
   At AbbVie, we believe that every patient deserves equal access to groundbreaking research. That's why we're committed to enhancing diversity and inclusion in our clinical trials. Join us in making a difference! Learn more: [Link] #ClinicalTrials #DiversityMatters #InclusiveResearch #HealthEquity
 Evaluation: {'appropriate_length': False, 'has_hashtags': True, 'professional_tone': True}, Score: 66.67%
  Test 2/4: tweet generation
Prompt: Write a motivational tweet for patients living with chronic illness, in the professional yet accessi...
 Generated in 7.70s (71 tokens)
   Living with a chronic illness can be challenging, but remember: you are not alone on this journey. Each day is an opportunity to take control and manage your health proactive